In [ ]:
# ==========================================
# 1. SZEGMENT: Könyvtárak és hardver beállítása
# ==========================================
import torch
from diffusers import DiffusionPipeline
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
size = 512 if device == "cuda" else 256


# ==========================================
# 2. SZEGMENT: Modellek egyszeri betöltése
# ==========================================
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=dtype,
    variant="fp16" if device == "cuda" else None
).to(device)

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype=dtype
).to(device)


# ==========================================
# 3. SZEGMENT: 1. kör (Szöveg -> Kép -> Szöveg)
# ==========================================
prompt_1 = "motorbike"
image_1 = pipe(prompt_1, num_inference_steps=1, guidance_scale=0.0, height=size, width=size).images[0]
display(image_1)

inputs_1 = processor(image_1, return_tensors="pt").to(device, dtype)
with torch.no_grad():
    out_1 = model.generate(**inputs_1, max_new_tokens=20)
caption_1 = processor.decode(out_1[0], skip_special_tokens=True)
print("1. képleírás:", caption_1)


# ==========================================
# 4. SZEGMENT: 2. kör (Új szöveg -> Kép -> Szöveg)
# ==========================================
image_2 = pipe(caption_1, num_inference_steps=1, guidance_scale=0.0, height=size, width=size).images[0]
display(image_2)

inputs_2 = processor(image_2, return_tensors="pt").to(device, dtype)
with torch.no_grad():
    out_2 = model.generate(**inputs_2, max_new_tokens=20)
caption_2 = processor.decode(out_2[0], skip_special_tokens=True)
print("2. képleírás:", caption_2)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]